# Extension du reseau Zipline — Clustering K-Means

**Auteur** : Semia BEN AMARA
**Date** : Mars 2026
**Approche** : Clustering K-Means (K=5) sur les facilities non couvertes par Zipline pour identifier 5 nouveaux hubs Standard complementaires

## Problematique
Les 6 hubs Zipline existants couvrent 73% des 2463 facilities de sante du Ghana (rayon 80 km). 665 facilities restent hors couverture. Ce notebook cherche a placer K=5 nouveaux hubs Standard pour maximiser la couverture.

## Donnees de depart
- `ghana_health_eda_final.csv` : 2463 facilities (colonne `Is_Covered` indique la couverture Zipline)
- `ghana_villages_eda_final.csv` : 8905 villages
- Contrainte : rayon max 80 km (Haversine)

## 1. Imports et chargement des donnees

In [ ]:
#Importation des données nettoyées
#On repart des fichiers "Master Clean" créés dans le notebook précédent.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import geodesic
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Chargement des données diagnostiquées
DATA_DIR = Path("../..") / "data" / "Final Group"
df_health = pd.read_csv(DATA_DIR / "ghana_health_eda_final.csv")

## 2. Filtrage des facilities non couvertes

In [ ]:
#: Isolation des "Zones Blanches"
#On ne veut placer de nouveaux hubs QUE là où Zipline ne couvre rien.
# On filtre les établissements non couverts (Is_Covered == 0)
white_zones = df_health[df_health['Is_Covered'] == 0][['Latitude', 'Longitude']]
print(f"Nombre de centres de santé restant à couvrir : {len(white_zones)}")

## 3. Recherche du K optimal (methode du coude + Silhouette)

In [ ]:
#Recherche du K Optimal (Méthode du Coude)
#C'est la preuve mathématique que K=5 est un bon compromis.
distortions = []
K_range = range(1, 12)

for k in K_range:
    kmeanModel = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeanModel.fit(white_zones)
    distortions.append(kmeanModel.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, distortions, 'bx-')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie (Distorsion)')
plt.title('Méthode du Coude : Trouver le point d\'inflexion')
plt.axvline(x=5, color='red', linestyle='--') # On marque le 5
plt.show()

In [ ]:
#Validation par le Score de Silhouette
#Pour confirmer que les clusters sont bien denses et séparés.
sil_scores = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(white_zones)
    score = silhouette_score(white_zones, km.labels_)
    sil_scores.append(score)

plt.figure(figsize=(10, 5))
plt.plot(range(2, 10), sil_scores, 'go-')
plt.title('Score de Silhouette par nombre de clusters')
plt.show()

## 4. Clustering K-Means K=5

In [ ]:
#Entraînement Final du Modèle (K=5)
# Application du K-Means final
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
white_zones['cluster'] = kmeans_final.fit_predict(white_zones[['Latitude', 'Longitude']])
new_hubs_coords = kmeans_final.cluster_centers_

print("Coordonnées des 5 nouveaux hubs optimisés :")
for i, hub in enumerate(new_hubs_coords):
    print(f"Hub {i+1} : {hub}")

## 5. Verification de la couverture

In [ ]:
#Calcul de la Nouvelle Couverture Totale
#On mesure l'impact de l'ajout de ces 5 hubs sur tout le territoire.
def check_total_coverage(row):
    p = (row['Latitude'], row['Longitude'])
    # Distance minimale par rapport à Zipline OU nos nouveaux hubs
    dists = [geodesic(p, h).km for h in new_hubs_coords]
    dists.append(row['Dist_Zipline_km'])
    return 1 if min(dists) <= 80 else 0

df_health['Full_Network_Covered'] = df_health.apply(check_total_coverage, axis=1)

final_pct = df_health['Full_Network_Covered'].mean() * 100
print(f"Couverture finale atteinte : {final_pct:.1f}%")

## 6. Carte interactive

In [ ]:
#On définit les 6 hubs existants et on mesure la distance pour chaque centre de santé
ZIPLINE_HUBS = [
    {"name": "Omenako", "loc": [6.2027, -0.4462]}, {"name": "Mpanya", "loc": [7.0545, -1.3855]},
    {"name": "Vobsi", "loc": [10.3470, -0.8035]}, {"name": "Sefwi Wiawso", "loc": [6.1820, -2.4845]},
    {"name": "Anum", "loc": [6.4885, 0.1625]}, {"name": "Kete Krachi", "loc": [7.7981, -0.0125]}
]

In [ ]:
# On s'assure que le nom est exactement celui utilisé dans la carte plus bas
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
white_zones['cluster'] = kmeans_final.fit_predict(white_zones[['Latitude', 'Longitude']])

# C'est cette variable qui manquait
new_hubs_coords = kmeans_final.cluster_centers_ 

print(f"Calcul terminé pour {len(new_hubs_coords)} hubs.")

In [ ]:
import folium
from folium.plugins import Fullscreen, HeatMap

# 1. Préparation des données d'impact
# Population couverte par la solution (Hubs Zipline + Nos 5 Hubs)
# On considère un village couvert s'il est à <= 80km d'un des 11 hubs
def is_village_covered(row):
    p = (row['Latitude'], row['Longitude'])
    d_zip = min([geodesic(p, h['loc']).km for h in ZIPLINE_HUBS])
    d_new = min([geodesic(p, nh).km for nh in new_hubs_coords])
    return 1 if min(d_zip, d_new) <= 80 else 0

#charger la dataset des villages
df_villages = pd.read_csv(DATA_DIR / "ghana_villages_eda_final.csv")

df_villages['Covered_Sol2'] = df_villages.apply(is_village_covered, axis=1)
total_pop_couverte = df_villages[df_villages['Covered_Sol2'] == 1]['Population'].sum()
pct_pop = (total_pop_couverte / df_villages['Population'].sum()) * 100

# 2. Configuration de la Carte
m_sol2 = folium.Map(location=[7.9, -1.0], zoom_start=7, tiles='CartoDB positron')

# Couleurs identiques au Notebook 1
COLOR_MAP = {
    'Hospital': '#E74C3C', 'Clinic': '#3498DB', 'CHPS': '#2ECC71', 
    'Health Centre': '#F1C40F', 'Other': '#95A5A6'
}

# Groupes de couches
fg_zipline = folium.FeatureGroup(name="Réseau Zipline Actuel", show=True)
fg_extension = folium.FeatureGroup(name="Extension DHG (5 Hubs K-Means)", show=True)
fg_health = folium.FeatureGroup(name="Établissements de Santé", show=True)
fg_heat = folium.FeatureGroup(name="Densité Population (Heatmap)", show=False)

# --- AJOUT RÉSEAU ZIPLINE ---
for h in ZIPLINE_HUBS:
    folium.Marker(h['loc'], icon=folium.Icon(color='black', icon='plane', prefix='fa')).add_to(fg_zipline)
    folium.Circle(h['loc'], radius=80000, color='black', fill=True, opacity=0.05).add_to(fg_zipline)

# --- AJOUT NOS NOUVEAUX HUBS (K-Means) ---
for i, coords in enumerate(new_hubs_coords):
    folium.Marker(location=[coords[0], coords[1]], 
                  icon=folium.Icon(color='red', icon='star', prefix='fa'),
                  popup=f"Hub Optimisé DHG {i+1}").add_to(fg_extension)
    folium.Circle(location=[coords[0], coords[1]], radius=80000, 
                  color='red', fill=True, opacity=0.1, dash_array='5, 10').add_to(fg_extension)

# --- AJOUT SANTÉ (5 COULEURS) ---
for _, row in df_health.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=2, color=COLOR_MAP.get(row['Category'], '#7F8C8D'),
        fill=True, fill_opacity=0.7
    ).add_to(fg_health)

# --- AJOUT HEATMAP POPULATION ---
HeatMap(df_villages[['Latitude', 'Longitude']].values, radius=10, blur=15).add_to(fg_heat)

# 3. LÉGANDE DÉTAILLÉE (Santé + Impact)
legend_html = f'''
     <div style="position: fixed; bottom: 30px; left: 30px; width: 260px; z-index:9999; 
     background-color: white; padding: 15px; border: 2px solid #E74C3C; border-radius: 10px; font-family: sans-serif; font-size:12px;">
     <b style="font-size:14px; color:#E74C3C;">IMPACT SOLUTION 2</b><br><br>
     <b>Performance :</b><br>
     Centres de sante : <b>{final_pct:.1f}%</b><br>
     Population couverte : <b>{pct_pop:.1f}%</b> ({total_pop_couverte/1e6:.2f}M hab.)<br>
     <hr>
     <b>Legende Sante :</b><br>
     <i style="background:{COLOR_MAP['Hospital']};width:10px;height:10px;display:inline-block"></i> Hopital 
     <i style="background:{COLOR_MAP['Clinic']};width:10px;height:10px;display:inline-block"></i> Clinique<br>
     <i style="background:{COLOR_MAP['CHPS']};width:10px;height:10px;display:inline-block"></i> CHPS 
     <i style="background:{COLOR_MAP['Health Centre']};width:10px;height:10px;display:inline-block"></i> Centre Sante<br><br>
     <i style="background:black;width:10px;height:10px;display:inline-block"></i> Hub Zipline 
     <i style="background:red;width:10px;height:10px;display:inline-block"></i> Hub DHG (Nouveau)
     </div>
     '''
m_sol2.get_root().html.add_child(folium.Element(legend_html))

# 4. Assemblage final
fg_zipline.add_to(m_sol2)
fg_extension.add_to(m_sol2)
fg_health.add_to(m_sol2)
fg_heat.add_to(m_sol2)
folium.LayerControl().add_to(m_sol2)
Fullscreen().add_to(m_sol2)

m_sol2.save('Solution_Extension_Finale.html')
m_sol2

## 7. Budget et dimensionnement

In [ ]:
# --- PARAMÈTRES FINANCIERS BASÉS SUR LE RAPPORT DHG 2026 ---
NB_HUBS = 5 

# Valeurs pour un Hub de type "Standard" (Source: Rapport Business Analyst)
CAPEX_UNITAIRE = 2000000       # Médiane retenue pour infrastructure + flotte 
OPEX_MENSUEL_UNITAIRE = 30000  # Moyenne staff + maintenance [cite: 165]
DRONES_PAR_HUB = 15            # Configuration standard (10-20 drones) 

# --- CALCULS ---
total_capex = NB_HUBS * CAPEX_UNITAIRE
total_opex_annuel = NB_HUBS * OPEX_MENSUEL_UNITAIRE * 12
total_drones = NB_HUBS * DRONES_PAR_HUB

# --- AFFICHAGE ---
print(f"--- BILAN FINANCIER : SOLUTION 2 (FULL STANDARD) ---")
print(f"Configuration : {NB_HUBS} Hubs Standards")
print(f"Flotte totale : {total_drones} Drones")
print(f"--------------------------------------------------")
print(f"INVESTISSEMENT INITIAL (CAPEX) : {total_capex:,.0f} $")
print(f"COÛT DE FONCTIONNEMENT (OPEX)  : {total_opex_annuel:,.0f} $ / an")
print(f"--------------------------------------------------")
print(f"Note : Chaque hub coûte {CAPEX_UNITAIRE:,.0f} $ à installer.")

##Conclusion

Analyse de l'Impact et Limites :
L’implémentation de ces 5 nouveaux hubs de type Standard transforme radicalement la logistique médicale au Ghana. Nous atteignons désormais un taux de couverture critique de 93 % pour les établissements de santé ainsi que pour la population totale.Cependant, malgré un investissement de 10 millions de dollars en CAPEX, l’analyse visuelle révèle une zone de rupture : une partie de l'Ouest du pays reste encore hors de portée du réseau de drones (zones blanches).
Problématique pour la suite :
Le déploiement uniforme de hubs standards à 2 millions de dollars l'unité  montre ses limites économiques pour couvrir ces derniers kilomètres. Est-il pertinent de financer une infrastructure lourde pour des zones de faible densité, ou devons-nous envisager une approche plus flexible ?

Prochaine étape (Notebook 3) :Nous explorerons une stratégie hybride. L'objectif sera de combler ces zones blanches de l'Ouest tout en optimisant la rentabilité globale du réseau grâce à une variation de la taille des hubs et de la flotte.

## 8. Corrections proposees (a valider par l'auteur)

| # | Element | Constat | Correction proposee | Pourquoi |
|---|---|---|---|---|
| S1 | `matplotlib.pyplot` | Non installe dans l'env `drone-ghana` | Remplacer par `plotly.express` | Stack projet = plotly + folium |
| S2 | Chemins CSV | Relatifs sans prefixe (CWD) | `Path("../..") / "data" / "Final Group"` | Structure repo standardisee |
| S3 | `ghana_villages_clean.csv` | Fichier non disponible dans le repo | Utiliser `ghana_villages_eda_final.csv` | CSV de base commun valide |
| S4 | CAPEX/hub = 2M$ | Bareme Standard uniquement | OK a conserver — le BA donne 2M$ (mediane) pour Standard | Coherence interne |
| S5 | Drones/hub = 15 | Valeur fixe sans justification | Ajouter une note de justification | BA : Standard = 10-20 drones |